# AI-Powered Gen Z Credit Card Product Innovation System
## Notebook 03 – Product Identity Classification

Description

This notebook develops a supervised machine learning model to classify Canadian credit card products into predefined product identities. A Random Forest classifier is trained using engineered product features and validated to predict the product identity of newly generated AI concepts. The trained model becomes part of the product evaluation pipeline.

Main Tasks
• Prepare classification dataset
• Train Random Forest classifier
• Model evaluation
• Save trained model
• Save feature order
• Save LabelEncoder

Output
• Trained Random Forest model
• Product identity classifier
• Feature order
• Label encoder

In [ ]:
#Phace 4
# 1. IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

#2. Import data

import pandas as pd

df = pd.read_csv("synthetic_credit_card_products_clustered_v2.csv")

print(df.shape)
df.head()


In [ ]:
#Select Features
feature_columns = [

    "welcome_bonus_value_cad",
    "annual_fee_cad",
    "minimum_income_requirement",
    "minimum_credit_score",
    "foreign_transaction_fee_percent",
    "purchase_interest_rate",
    "base_reward_rate",
    "cashback_feature_level",
    "lifestyle_feature_level",
    "travel_feature_level",
    "digital_feature_level",
    "subscription_feature_level",
    "sustainability_feature_level",
    "financial_wellness_feature_level",
    "security_protection_feature_level",
    "premium_benefits_feature_level",
   

]

In [ ]:
# Create X and y
X = df[feature_columns]

y = df[ "cluster_name"]

In [ ]:
# Encode Labels
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

y_encoded = encoder.fit_transform(y)

print(encoder.classes_)

In [ ]:
# Train/Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(X_train.shape)
print(X_test.shape)

In [ ]:
# Train Random Forest
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)

print("Model trained successfully")

In [ ]:
# Make Predictions
y_pred = rf.predict(X_test)

In [ ]:
# Evaluate Model
from sklearn.metrics import classification_report, accuracy_score

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy, 4))
print("\nClassification Report:\n")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=[str(label) for label in encoder.classes_]
    )
)

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(10,8))

plt.imshow(cm)

plt.colorbar()

plt.xticks(
    range(len(encoder.classes_)),
    encoder.classes_,
    rotation=45
)

plt.yticks(
    range(len(encoder.classes_)),
    encoder.classes_
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.title("Confusion Matrix")

plt.tight_layout()

plt.show()

In [ ]:
#Feature Importance
importance = pd.DataFrame({

    "Feature": X.columns,
    "Importance": rf.feature_importances_

})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance)

In [ ]:
# Visualize Top Features
top_features = importance.head(10)

plt.figure(figsize=(10,6))

plt.barh(
    top_features["Feature"],
    top_features["Importance"]
)

plt.title("Top 10 Most Important Features")

plt.tight_layout()

plt.show()

# Product Classification Results

A Random Forest classifier was trained to predict the primary product type using pricing, rewards, digital, sustainability, financial wellness, and clustering features.

The model achieved a high classification accuracy, indicating that the synthetic dataset contains meaningful relationships between product attributes and product categories.

Feature importance analysis revealed that product intelligence scores such as travel focus, cashback focus, digital focus, sustainability focus, and financial wellness scores were among the strongest predictors of product type.

These results demonstrate that machine learning can successfully classify credit card products based on their characteristics and can support future product intelligence, market analysis, and product innovation initiatives.

In [ ]:
sample = X_test.iloc[[66]]

prediction = rf.predict(sample)

print("Predicted Class:")
print(encoder.inverse_transform(prediction))

In [ ]:
actual = encoder.inverse_transform([y_test[66]])

print("Actual:", actual)
print("Predicted:", encoder.inverse_transform(prediction))

In [ ]:
# Check the exact feature order used during training
print(rf.feature_names_in_)

In [ ]:
new_card = pd.DataFrame({

    "annual_fee_cad":[0],
    "minimum_income_requirement":[12000],
    "minimum_credit_score":[120],
    "foreign_transaction_fee_percent":[0],
    "purchase_interest_rate":[21],
    "base_reward_rate":[2.5],
    "travel_feature_level":[0],
    "cashback_feature_level":[40],
    "digital_feature_level":[40],
    "subscription_feature_level":[10],
    "lifestyle_feature_level":[20],
    "sustainability_feature_level":[50],
    "financial_wellness_feature_level":[25],
    "security_protection_feature_level":[50],
    "welcome_bonus_value_cad":[1200],
    "premium_benefits_feature_level":[0],
   
})

In [ ]:
# Match the exact training feature order
new_card = new_card.reindex(columns=rf.feature_names_in_)

# Predict encoded class
prediction = rf.predict(new_card)

# Convert encoded class to product identity name
predicted_identity = encoder.inverse_transform(prediction)[0]

print("Predicted Product Identity:", predicted_identity)

In [ ]:
# Predict
prediction = rf.predict(new_card)
probabilities = rf.predict_proba(new_card)

# Decode prediction
predicted_identity = encoder.inverse_transform(prediction)[0]
confidence = probabilities.max() * 100

print("Predicted Product Identity:", predicted_identity)
print(f"Confidence: {confidence:.2f}%")

In [ ]:
import joblib

# Save classification model
joblib.dump(rf, "credit_card_classification_model.pkl")

# Save label encoder
joblib.dump(encoder, "product_type_label_encoder.pkl")

# Save feature columns
joblib.dump(feature_columns, "classification_feature_columns.pkl")

print("Model files saved successfully.")

In [ ]:
loaded_model = joblib.load(
    "credit_card_classification_model.pkl"
)

loaded_features = joblib.load(
    "classification_feature_columns.pkl"
)

new_card = new_card.reindex(columns=loaded_features)

prediction = loaded_model.predict(new_card)

print("Predicted Product Identity:", prediction[0])